# Introduction

This Jupyter Notebook provides a complete pipeline for household energy consumption analysis. It includes data loading, preprocessing, feature engineering, database operations, model development, visualization, and optimization recommendations.

Start by installing the needed libraries and dependencies below!

In [ ]:
%pip install pandas numpy sqlalchemy scikit-learn plotly requests

# Data Collection and Preprocessing
Import required libraries, load household energy consumption dataset, handle missing values, and perform initial data cleaning using Pandas and NumPy.

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np

# Load household energy consumption dataset
# Assuming the dataset is in CSV format and named 'household_energy_consumption.csv'
data = pd.read_csv('household_energy_consumption.csv')

# Display the first few rows of the dataset
data.head()

# Handle missing values
# Fill missing values with the mean of the respective columns
data.fillna(data.mean(), inplace=True)

# Perform initial data cleaning
# Convert timestamp column to datetime format
data['timestamp'] = pd.to_datetime(data['timestamp'])

# Generate derived features
data['hour'] = data['timestamp'].dt.hour
data['day_of_week'] = data['timestamp'].dt.dayofweek
data['is_weekend'] = data['day_of_week'] >= 5

# Display the cleaned dataset
data.head()

# Feature Engineering
Create time-based features, calculate neighborhood aggregates, and prepare geospatial data linking using Pandas and custom functions.

In [ ]:
# Feature Engineering

# Create time-based features
data['month'] = data['timestamp'].dt.month
data['day'] = data['timestamp'].dt.day

# Calculate neighborhood aggregates
# Assuming 'neighborhood' column exists in the dataset
neighborhood_aggregates = data.groupby(['neighborhood', 'timestamp']).agg({
    'energy_consumption': ['mean', 'sum', 'max', 'min']
}).reset_index()

# Flatten the multi-level column names
neighborhood_aggregates.columns = ['_'.join(col).strip() for col in neighborhood_aggregates.columns.values]

# Merge neighborhood aggregates back to the main dataset
data = pd.merge(data, neighborhood_aggregates, left_on=['neighborhood', 'timestamp'], right_on=['neighborhood_', 'timestamp_'], how='left')

# Prepare geospatial data linking
# Assuming 'latitude' and 'longitude' columns exist in the dataset
data['coordinates'] = list(zip(data['latitude'], data['longitude']))

# Display the dataset with new features
data.head()

# Database Setup and Operations
Initialize SQLite database, create tables for structured data storage, and implement data insertion/retrieval operations using SQLAlchemy.

In [ ]:
# Database Setup and Operations

# Import SQLAlchemy and other required libraries
from sqlalchemy import create_engine, Column, Integer, Float, String, DateTime
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker

# Initialize SQLite database
engine = create_engine('sqlite:///energy_consumption.db')
Base = declarative_base()

# Define the HouseholdEnergy table
class HouseholdEnergy(Base):
    __tablename__ = 'household_energy'
    id = Column(Integer, primary_key=True, autoincrement=True)
    timestamp = Column(DateTime)
    neighborhood = Column(String)
    energy_consumption = Column(Float)
    hour = Column(Integer)
    day_of_week = Column(Integer)
    is_weekend = Column(Integer)
    month = Column(Integer)
    day = Column(Integer)
    latitude = Column(Float)
    longitude = Column(Float)
    coordinates = Column(String)
    energy_mean = Column(Float)
    energy_sum = Column(Float)
    energy_max = Column(Float)
    energy_min = Column(Float)

# Create tables in the database
Base.metadata.create_all(engine)

# Create a session
Session = sessionmaker(bind=engine)
session = Session()

# Insert data into the database
for index, row in data.iterrows():
    household_energy = HouseholdEnergy(
        timestamp=row['timestamp'],
        neighborhood=row['neighborhood'],
        energy_consumption=row['energy_consumption'],
        hour=row['hour'],
        day_of_week=row['day_of_week'],
        is_weekend=row['is_weekend'],
        month=row['month'],
        day=row['day'],
        latitude=row['latitude'],
        longitude=row['longitude'],
        coordinates=str(row['coordinates']),
        energy_mean=row['energy_consumption_mean'],
        energy_sum=row['energy_consumption_sum'],
        energy_max=row['energy_consumption_max'],
        energy_min=row['energy_consumption_min']
    )
    session.add(household_energy)

# Commit the session to save the data
session.commit()

# Retrieve and display data from the database
retrieved_data = session.query(HouseholdEnergy).all()
for record in retrieved_data[:5]:  # Display first 5 records
    print(record.__dict__)

# Model Development
Implement Random Forest Regressor, perform train-test split, conduct hyperparameter tuning, and evaluate model performance using scikit-learn.

In [ ]:
# Model Development

# Import necessary libraries for machine learning
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Define the feature columns and target column
feature_columns = ['hour', 'day_of_week', 'is_weekend', 'month', 'day', 'energy_mean', 'energy_sum', 'energy_max', 'energy_min']
target_column = 'energy_consumption'

# Split the data into features (X) and target (y)
X = data[feature_columns]
y = data[target_column]

# Perform train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize the Random Forest Regressor
rf = RandomForestRegressor(random_state=42)

# Define the hyperparameters grid for tuning
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# Perform Grid Search with Cross-Validation
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=3, n_jobs=-1, verbose=2)
grid_search.fit(X_train, y_train)

# Get the best estimator from the grid search
best_rf = grid_search.best_estimator_

# Make predictions on the test set
y_pred = best_rf.predict(X_test)

# Evaluate the model performance
rmse = mean_squared_error(y_test, y_pred, squared=False)
r2 = r2_score(y_test, y_pred)

# Print the evaluation metrics
print(f'Root Mean Squared Error (RMSE): {rmse}')
print(f'R-squared (R2): {r2}')

# Visualization Components
Create interactive visualizations using Plotly, implement Google Maps integration for geospatial visualization, and prepare data for Tableau export.

In [ ]:
# Visualization Components

# Import necessary libraries for visualization
import plotly.express as px
import plotly.graph_objects as go
import json
import requests

# Create interactive time-series plot using Plotly
fig = px.line(data, x='timestamp', y='energy_consumption', title='Energy Consumption Over Time')
fig.show()

# Create a scatter plot with geospatial data using Plotly
fig = px.scatter_mapbox(data, lat='latitude', lon='longitude', color='energy_consumption',
                        size='energy_consumption', size_max=15, zoom=10,
                        mapbox_style='carto-positron', title='Geospatial Energy Consumption')
fig.show()

# Prepare data for Tableau export
# Export the cleaned and processed data to a CSV file
data.to_csv('processed_energy_data.csv', index=False)

# Google Maps Integration for geospatial visualization
# Assuming you have a Google Maps API key
api_key = 'YOUR_GOOGLE_MAPS_API_KEY'

# Function to create a Google Maps URL with markers
def create_google_maps_url(data):
    base_url = "https://maps.googleapis.com/maps/api/staticmap?"
    center = "center=0,0"
    zoom = "zoom=2"
    size = "size=800x800"
    maptype = "maptype=roadmap"
    markers = "&".join([f"markers=color:red%7Clabel:{i}%7C{row['latitude']},{row['longitude']}" for i, row in data.iterrows()])
    key = f"key={api_key}"
    return f"{base_url}{center}&{zoom}&{size}&{maptype}&{markers}&{key}"

# Generate the Google Maps URL
google_maps_url = create_google_maps_url(data)

# Display the Google Maps URL
print("Google Maps URL:", google_maps_url)

# Display the Google Maps image
response = requests.get(google_maps_url)
img = response.content

# Save the image
with open('google_maps_image.png', 'wb') as f:
    f.write(img)

# Display the image using Plotly
fig = go.Figure()
fig.add_layout_image(
    dict(
        source='google_maps_image.png',
        xref="x",
        yref="y",
        x=0,
        y=0,
        sizex=1,
        sizey=1,
        sizing="stretch",
        opacity=1,
        layer="below"
    )
)
fig.update_layout(
    title='Geospatial Energy Consumption (Google Maps)',
    xaxis=dict(visible=False),
    yaxis=dict(visible=False)
)
fig.show()

# Optimization Engine
Develop algorithms for generating energy optimization recommendations based on prediction results and historical patterns.

In [ ]:
# Optimization Engine

# Function to generate optimization recommendations
def generate_optimization_recommendations(data, predictions):
    recommendations = []
    for index, row in data.iterrows():
        predicted_consumption = predictions[index]
        current_consumption = row['energy_consumption']
        
        # Calculate the difference between predicted and current consumption
        consumption_diff = predicted_consumption - current_consumption
        
        # Generate recommendation based on the difference
        if consumption_diff > 0:
            recommendation = f"Reduce energy usage by {consumption_diff:.2f} units to optimize consumption."
        else:
            recommendation = f"Energy usage is optimal. No changes needed."
        
        recommendations.append(recommendation)
    
    return recommendations

# Generate optimization recommendations based on predictions
optimization_recommendations = generate_optimization_recommendations(X_test, y_pred)

# Add recommendations to the test dataset
X_test['recommendations'] = optimization_recommendations

# Display the first few rows with recommendations
X_test.head()

# Integration Testing
Test the complete pipeline from data processing to visualization, validate outputs, and ensure system components work together seamlessly.

In [ ]:
# Integration Testing

# Function to test the complete pipeline
def test_pipeline(data):
    # Step 1: Data Processing
    data.fillna(data.mean(), inplace=True)
    data['timestamp'] = pd.to_datetime(data['timestamp'])
    data['hour'] = data['timestamp'].dt.hour
    data['day_of_week'] = data['timestamp'].dt.dayofweek
    data['is_weekend'] = data['day_of_week'] >= 5
    data['month'] = data['timestamp'].dt.month
    data['day'] = data['timestamp'].dt.day
    neighborhood_aggregates = data.groupby(['neighborhood', 'timestamp']).agg({
        'energy_consumption': ['mean', 'sum', 'max', 'min']
    }).reset_index()
    neighborhood_aggregates.columns = ['_'.join(col).strip() for col in neighborhood_aggregates.columns.values]
    data = pd.merge(data, neighborhood_aggregates, left_on=['neighborhood', 'timestamp'], right_on=['neighborhood_', 'timestamp_'], how='left')
    data['coordinates'] = list(zip(data['latitude'], data['longitude']))

    # Step 2: Database Operations
    engine = create_engine('sqlite:///energy_consumption.db')
    Base.metadata.create_all(engine)
    Session = sessionmaker(bind=engine)
    session = Session()
    for index, row in data.iterrows():
        household_energy = HouseholdEnergy(
            timestamp=row['timestamp'],
            neighborhood=row['neighborhood'],
            energy_consumption=row['energy_consumption'],
            hour=row['hour'],
            day_of_week=row['day_of_week'],
            is_weekend=row['is_weekend'],
            month=row['month'],
            day=row['day'],
            latitude=row['latitude'],
            longitude=row['longitude'],
            coordinates=str(row['coordinates']),
            energy_mean=row['energy_consumption_mean'],
            energy_sum=row['energy_consumption_sum'],
            energy_max=row['energy_consumption_max'],
            energy_min=row['energy_consumption_min']
        )
        session.add(household_energy)
    session.commit()

    # Step 3: Model Training and Prediction
    feature_columns = ['hour', 'day_of_week', 'is_weekend', 'month', 'day', 'energy_mean', 'energy_sum', 'energy_max', 'energy_min']
    target_column = 'energy_consumption'
    X = data[feature_columns]
    y = data[target_column]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    rf = RandomForestRegressor(random_state=42)
    param_grid = {
        'n_estimators': [100, 200, 300],
        'max_depth': [10, 20, 30],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4]
    }
    grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=3, n_jobs=-1, verbose=2)
    grid_search.fit(X_train, y_train)
    best_rf = grid_search.best_estimator_
    y_pred = best_rf.predict(X_test)
    rmse = mean_squared_error(y_test, y_pred, squared=False)
    r2 = r2_score(y_test, y_pred)

    # Step 4: Visualization
    fig = px.line(data, x='timestamp', y='energy_consumption', title='Energy Consumption Over Time')
    fig.show()
    fig = px.scatter_mapbox(data, lat='latitude', lon='longitude', color='energy_consumption',
                            size='energy_consumption', size_max=15, zoom=10,
                            mapbox_style='carto-positron', title='Geospatial Energy Consumption')
    fig.show()
    google_maps_url = create_google_maps_url(data)
    response = requests.get(google_maps_url)
    img = response.content
    with open('google_maps_image.png', 'wb') as f:
        f.write(img)
    fig = go.Figure()
    fig.add_layout_image(
        dict(
            source='google_maps_image.png',
            xref="x",
            yref="y",
            x=0,
            y=0,
            sizex=1,
            sizey=1,
            sizing="stretch",
            opacity=1,
            layer="below"
        )
    )
    fig.update_layout(
        title='Geospatial Energy Consumption (Google Maps)',
        xaxis=dict(visible=False),
        yaxis=dict(visible=False)
    )
    fig.show()

    # Step 5: Optimization Recommendations
    optimization_recommendations = generate_optimization_recommendations(X_test, y_pred)
    X_test['recommendations'] = optimization_recommendations

    # Return the evaluation metrics and a sample of the test set with recommendations
    return rmse, r2, X_test.head()

# Run the integration test
rmse, r2, sample_recommendations = test_pipeline(data)

# Print the results
print(f'Integration Test - Root Mean Squared Error (RMSE): {rmse}')
print(f'Integration Test - R-squared (R2): {r2}')
print('Sample Recommendations:')
print(sample_recommendations)